In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, lit, current_timestamp
from datetime import datetime
import requests
import pandas as pd
import io

spark.sql("USE  CATALOG scottish_housing_ws")
spark.sql("USE SCHEMA bronze")

In [0]:
url = "https://www.bankofengland.co.uk/boeapps/database/_iadb-fromshowcolumns.asp?csv.x=yes&SeriesCodes=LPMVTVN&CSVF=TN&UsingCodes=Y&Datefrom=01/Jan/2004&Dateto=01/Jan/2030"

headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(url, headers=headers)
response.raise_for_status()

df_pd = pd.read_csv(io.StringIO(response.text))
print(df_pd.shape)
print(df_pd.head)
           

In [0]:
df_bronze = spark.createDataFrame(df_pd)
(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze.boe_mortgage_approvals")
)

In [0]:
spark.sql("SELECT * FROM bronze.boe_mortgage_approvals LIMIT 5").show()
spark.sql("SELECT COUNT(*) FROM bronze.boe_mortgage_approvals").show()

In [0]:
# silver.boe_mortgage_approvals
# 267 rows, Jan 2004 to Mar 2026
# UK-wide mortgage approvals count, monthly
# report_date (date), year_month (string), mortgage_approvals_count (long)